# W2 Homework — An Evalset of Your Own

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week02/W2_hw_build_evalset.ipynb)

**Goal.** Design an eight-item code-graded evalset in a domain you choose, then show
with numbers that a chain-of-thought prompt beats answer-only prompting on it.

An eval is only as good as its items: easy items measure nothing, leaked answers
measure leakage (the design rules return formally in Ch. 5).

The path: setup → the pattern on a worked three-item example → your evalset ✍️ →
baseline vs. your CoT prompt, measured ✍️ → completion.

*Runtime:* ~40 minutes. Due before the W3 session. Reference answers:
`labs/checkpoints/week02/solution.py`, published after the homework deadline.


## 1. Setup

Same standard as the labs: install, paste your key, run the helpers.

*Do:* run the three cells; the last must print `ready`.


In [ ]:
%pip install -q "aisuite[openai,anthropic]"


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # alt: "anthropic:claude-haiku-4-5"


In [ ]:
import aisuite

client = aisuite.Client()


def ask(prompt, system=None, temperature=0.0):
    """Single prompt -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=MODEL, messages=messages,
                                              temperature=temperature)
    return response.choices[0].message.content


print(ask("Reply with exactly: ready"))


## 2. The Pattern — a Worked Example

Three weekday-arithmetic items and the whole measuring machinery: each item is a
question plus a short checkable answer; the grader is a substring check on the
reply; `run_eval` scores a prompt template over the set. Item 3 is the deliberately
hard one — two steps (reduce 45 mod 7, then advance), where answer-only prompting
tends to slip.

*Do:* run the cell and read the per-item lines: which item fails under the baseline
template, and what does its wrong answer look like?


In [ ]:
EXAMPLE_EVALSET = [
    {"question": "If today is Wednesday, what day of the week will it be in 2 days?",
     "answer": "Friday"},
    {"question": "If today is Saturday, what day of the week was it 3 days ago?",
     "answer": "Wednesday"},
    {"question": "If today is Monday, what day of the week will it be in 45 days?",
     "answer": "Thursday"},
]

BASELINE_PROMPT = "Answer with the weekday name only.\n\nQuestion: {question}"


def run_eval(prompt_template, evalset, label=""):
    """Scores a prompt template over an evalset; returns the number correct."""
    correct = 0
    for item in evalset:
        reply = ask(prompt_template.format(question=item["question"]))
        ok = item["answer"].lower() in reply.lower()
        correct += ok
        print(f"{'PASS' if ok else 'FAIL':4}  {item['question'][:55]:55}  -> {reply.splitlines()[-1][:40]}")
    print(f"{label} score: {correct}/{len(evalset)}\n")
    return correct


example_baseline = run_eval(BASELINE_PROMPT, EXAMPLE_EVALSET, "example baseline")


## 3. Your Evalset ✍️

Replace the starter with **at least eight items in one domain of your own** — not
weekdays. Requirements:

- one domain, short checkable answers (a number, a single word, a short string);
- at least **two multi-step items** — a correct answer requires chaining two
  operations (these are the items that will separate your two prompts);
- no item's answer may appear anywhere in your prompt template (Section 4) — that
  would measure leakage, not ability — any worked examples inside your template
  must use items that are not in the evalset.

Hints — the ingredients:
1. Serviceable domains: unit conversion (minutes↔hours+minutes), date arithmetic,
   Roman numerals, letter counting in words, small percentage problems.
2. Copy the dict shape of `EXAMPLE_EVALSET` exactly: `{"question": ..., "answer": ...}`.
3. A multi-step recipe: take an easy item and add one conversion in front of it
   ("2 hours and 15 minutes" instead of "135 minutes").
4. Check your own answers by hand — a wrong golden answer poisons every run.
5. Prefer answers of two or more characters (a word, a multi-digit number): the
   grader looks for the golden answer inside the reply, and one-character answers
   match too easily.


In [ ]:
### FILL IN (START) ###
MY_EVALSET = list(EXAMPLE_EVALSET)   # starter only — replace with >= 8 items of your own domain
### FILL IN (END) ###

print(f"{len(MY_EVALSET)} items")
for item in MY_EVALSET[:3]:
    print("-", item["question"], "->", item["answer"])


## 4. Baseline, then Your CoT Prompt ✍️

The baseline template demands the answer only. Write `IMPROVED_PROMPT` so the model
works through the steps before answering. Requirements: it must keep the
`{question}` placeholder, instruct step-by-step working, and demand the final line
in a fixed form the grader can find (the answer word or number on the last line).

Target: **improved ≥ 7/8 and strictly above the baseline** on your set.

*Do:* run baseline first, read its failures, then write the improved template and
re-run. One template change per run — attribute your delta.


In [ ]:
my_baseline = run_eval(BASELINE_PROMPT, MY_EVALSET, "baseline")

In [ ]:
### FILL IN (START) ###
IMPROVED_PROMPT = BASELINE_PROMPT   # starter only — write your CoT template
### FILL IN (END) ###

my_improved = run_eval(IMPROVED_PROMPT, MY_EVALSET, "improved")
TARGET = len(MY_EVALSET) - 1


## 5. Completion Check

All rows must read `PASS` before submission; grading checks these structural facts,
never which domain you chose.


In [ ]:
import re

questions = [item["question"] for item in MY_EVALSET]
completion = {
    "evalset has >= 8 items": len(MY_EVALSET) >= 8,
    "items are unique": len(set(questions)) == len(questions),
    "own domain (not the weekday starter)":
        not any("day of the week" in q for q in questions),
    "no golden answer leaked into the template":
        all(len(str(item["answer"])) < 2
            or not re.search(rf"\b{re.escape(str(item['answer']).lower())}\b",
                             IMPROVED_PROMPT.lower())
            for item in MY_EVALSET),
    "improved prompt differs from baseline": IMPROVED_PROMPT != BASELINE_PROMPT,
    f"improved >= {TARGET}/{len(MY_EVALSET)}": my_improved >= TARGET,
    "improved > baseline": my_improved > my_baseline,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nHOMEWORK COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")
